# Datamine SURCAL Process Example

This notebook demonstrates how to configure and run the **`surcal`** process wrapper in `dmstudio`.

## Process Description

## SURCAL

This process computes provisional and/or adjusted coordinates of all observed survey stations contained in an input file of reduced observations such as output from process [SUROBS](<surobs.md>).

The coordinate computation method can be selected automatically or you can choose one. Checks are made to ensure enough observations have been taken to complete coordinate computation. If adjustments are required **SURCAL** allows the computation of a rigorous solution for each coordinate including calculation of the standard error of unknown values.

If new stations exist in the input survey station file, a recalculation of coordinates can take place and the user may allow the process to trace and recalculate the survey base-lines that have been established from that point. Where observations have been taken between fixed points, the misclosures of coordinates, distances and bearing are reported to the screen, printer and optionally to the output file of new survey station coordinates.

The method of coordinate computation may be summarized as:

* Automatic selection of computation method

* Bearing and distance method

* Angle and distance method

* Resection by angles

* Resection by distances

* Distance intersection

* Bearing intersection

* Angle intersection

If a combination of angle and distance measurements has taken place to fix one or more new stations from two or more existing stations, **SURCAL** can be used to compute the provisional coordinates of the new stations by one of the method listed above. When automatic mode is selected the information available for a particular new station is processed and the best computation method is used. The automatic selection of a computation method obeys a hierarchical network which is explained below.

Coordinate adjustment is optionally carried out when redundant measurements are available. Weights corresponding to the error fields such as **HZERR** , **VAERR** , **HDERR** can be applied onto the computations. Adjustments are based on a rigorous least square error analysis. An error ellipse is also evaluated and reported. Reiteration of the adjustment procedure is possible by re-assessment of observation standard errors. When adjustments are required, the user is prompted to confirm further reiterations. If reiteration is confirmed, then the weights to be applied are multiplied by the standard error of unknown values.

* When intersection methods are used, the measurements taken from fixed stations to the left of the fixed base-line, when looking towards the new station, must occur first in the input file.

* When parameters @**WEIGHT** and @**ADJUST** are both set to one, coordinate adjustments are processed with weighting. Weights assigned to the elevation differences are computed according to the inverse of the slope distance error. Any standard error of the elevation difference is not used in this case.

* Checks are made for inconsistent angle measurements (for example, vertical angles readings of more than 90 , WCB and horizontal angle readings of more than 360 or less than 0 ). For these cases the records are ignored.

* An important note refers to the precision of the coordinates handled in **SUROBS**. Although all variables are stored and handled by Double-Precision type variables and arrays, input and output measurements and coordinates are held in single precision type variables. In order to avoid output problems with values over 7 digits long, the use of parameters @**LOXORIG** , @**LOYORIG** and @**LOZORIG** is strongly recommended. Output display and report can be controlled by parameter @**NDEC**.

* Iterative computation of 2 (standard error of unknown values) is processed by displaying the most recently 2 value computed and prompting the user for further iteration. A response of <enter> or any numerical value proceeds to a new computation with the default value or the input value respectively. A response of <!> terminates the iterative computation and the final adjusted values are displayed. When operating through a macro the termination response <!> must be replaced by a <-1> response.

### File Handling

There are three compulsory files required by **SURCAL**. These may be summarized as:

* &**IN** : an input file containing the reduced observations as output from process SUROBS (reduction of observed survey measurements to mean angles and bearings with distances optionally reduced to a grid reference plane). If observation errors are present, then these may be optionally used for weighting in the adjustment process.

* &**CONTROL** : a file containing the coordinates and identification of established survey stations that will be used in the survey (compulsory).

* &**OUT** : a file containing the coordinated survey stations with reduced base line data and coordinate adjustment.

### Field Handling

All fields of the input files are compulsory. If the input observations file &IN is an output from process **SUROBS** no modifications are needed. A summary of the required fields is:

## &IN |  &CONTROL |  &OUT

---|---|---

## *INSTSTN *INSTHT *RO *TARGET *HZA *WCB *QB *WCBERR *VA *HDIST *HDERR *RDIST *VDIFF *VDERR *PLANE *FACTOR *REFRACT *ERRFLAG |  *STATION *X *Y *Z *RO *WCB *QB *HDIST *VDIFF *PLANE *FACTOR *REFRACT *HDERR *WCBERR *VAERR *ERRFLAG *LOXORIG *LOYORIG *LOZORIG |  *STATION *X *Y *Z *RO *WCB *QB *HDIST *VDIFF *PLANE *FACTOR *REFRACT *HDERR *WCBERR *VAERR *ERRFLAG *LOXORIG *LOYORIG *LOZORIG

Stations are identified by field * **STATION** in input file &**CONTROL** and fields * **INSTSTN** , * **TARGET** and * **RO** in input file &**IN** and output field &**OUT**. These must be alphanumeric and eight characters in length. Field QB refers to the quadrant bearing and must be twelve characters in length. All the remaining fields are numeric.

### Adjustments

The adjustment process is incorporated in **SURCAL** and may use weights as defined in the measurement error fields. This is achieved by the use of both **SUROBS** and **SURCAL** processes**. SUROBS** reduces repetitive survey observations into a mean value and an error (which could be the standard error of the mean or the range of the observations). The reduced observations are then fed into **SURCAL** to compute new survey station coordinates. **SURCAL** includes two stages of coordinate computation, a provisional computation and an adjustment process. A summary of the adjustment procedure is shown in Figure 1.

The provisional coordinate computation stage mentioned in Figure 1 refers to a network of hierarchical procedures to compute the coordinates when parameter @**CALCTYPE** is set to 0 (automatic mode). The preference in the network is given to coordinate computations methods in which a lower error level can be normally assumed. Therefore, bearing and distance method is preferred over bearing intersection, which in turn is preferred over distance intersection and so on. A summary of the hierarchical network is shown in Figure 2.

Computation by the hierarchical network gives one set of coordinates to each new station, flagging the results as "unique" or "non-unique" solution. The stations with non-unique solutions are those where adjustments are necessary; however, the adjustment process includes all new stations in one set of algebraic equations where the residuals (errors) are minimised and the most probable coordinates of each new station are computed. The solution of the algebraic equations includes inverse matrix computation. Due to the precision required by the adjustment process the L.U. decomposition method is used for inverse matrix computation (L.U. stands for lower and upper triangular decomposition) . This method uses Crout's algorithm.

It is important to note that **SURCAL** computes 3-D adjustments, therefore processing X, Y and Z coordinates. Details of the basic theory, variance analysis and weighting of physically dissimilar quantities used in **SURCAL** can be found in the article "Adjustment of Control Networks" published in the Chartered Surveyor magazine, of January, 1970.

### Error Ellipse and Reiterations of 2

The standard error of unknown values ( 2) for the coordinate calculation of each new station is computed from the coordinate adjustment procedure. After computing the adjusted coordinates, the error ellipse values are also calculated in SURCAL so that an assessment of the reliability of the newly-computed coordinates. At the same time, the computed value of 2 is displayed and the user is prompted for further iterations. Each iteration consists of applying the computed 2 to the algebraic equations, so that an improved new value of 2 can be computed. Any departure from the unity may imply in gross estimation errors and the user is warned about it. The most recently computed 2 value is displayed at the end of each iteration. To terminate the iterations, enter <!>. Refer to the Example section for more details on error ellipse reporting and iterative computation of 2.

### Input Files:

* **in**:
  Input file of reduced survey observations. This file will contain the following fields
  ((N) denotes Numeric, (A,8) denotes Alphanumeric field type and length):- **INSTSTN**
  (A,8) Survey station identifier for the instrument location. **INSTHT** (N) Instrument
  height. (Negative for instruments set below the survey station). **RO** (A,8) Survey
  station identifier for the reference object survey station. **TARGET** (A,8) Identifier
  of the survey station located. **HZA** (N) Mean horizontal angle measurement made to the
  target station. **WCB** (N) Whole Circle Bearing from **INSTSTN** to **TARGET**. **QB**
  (A,12)Quadrant Bearing **INSTSTN** to **TARGET**. **WCBERR** (N) Mean standard error or
  range of the measurements taken to establish the Whole Circle Bearing **INSTSTN** \-
  **TARGET**. **VA** (N) Mean vertical angle measurement made to the target station.
  **HDIST** (N) Mean horizontal distance from INSTSTN to TARGET. **HDERR** (N) Mean
  standard error or range of horizontal distances from slope distances and vertical
  angles. **RDIST** (N) Reference plane distance from **INSTSTN** to **TARGET** as
  computed from the horizontal distance **HDIST**. **VDIFF** (N) Mean difference in
  elevation from **INSTSTN** to **TARGET**. **VDERR** (N) Mean standard error or range of
  computed height differences from **INSTSTN** to **TARGET**. **PLANE** (N) Reference
  plane used to compute **RDIST** from **HDIST**. If absent, **RDIST** = **HDIST**.
  **FACTOR** (N) Scale factor used to compute **RDIST** after reduction to **PLANE**. The
  default must be 1. **REFRACT** (N) Coefficient of refraction used to adjust vertical
  angles where single measurements are made. **ERRFLAG** (N) Error flag field. Four digits
  may be set as follows:- ABCD for example
  Required=No

### Output Files:

* **out** (Undefined):
  Output file of newly coordinated survey stations. This file will contain the following

* **fields** (- **STATION** (A,8) Survey station identifier. **X** (N) X coordinate value.):
  **Y** (N) Y coordinate value. **Z** (N) Z coordinate value. **RO** (A,8) Identity of
  station used to locate **STATION**. **WCB** (N) Whole Circle Bearing from **STATION** to
  **RO**. Referred to as azimuth elsewhere in your application. **QB** (A,12)Quadrant
  Bearing **STATION** to **RO**. (such as N 45.0000 E). **HDIST** (N) Horizontal distance
  from **STATION** to **RO**. **RDIST** (N) Reduced distance from **STATION** to **RO** ,
  as computed at **PLANE** elevation. **VDIFF** (N) Vertical difference in height from
  **STATION** to **RO**. **PLANE** (N) Elevation to which **HDIST** has reduced to compute
  **RDIST**. If absent (-), no further reduction has been computed. **FACTOR** (N) Scale
  factor used to compute RDIST. **REFRACT** (N) Coefficient of refraction used to compute
  **VDIFF** where only a single forward vertical angle is used. **HDERR** (N) Mean
  standard error or range horizontal distances from **RO** to **STATION**. **WCBERR** (N)
  Mean standard error or range of horizontal angles used in computing the bearing from
  **RO** to **STATION**. **VAERR** (N) Mean standard error or range of vertical angles
  used in computing **VDIFF** from **RO** to **STATION**. **ERRFLAG** (N) Flag to identify
  when a measurement tolerance is exceeded. **LOXORIG** (N) Implicit local X origin field.
  **LOYORIG** (N) Implicit local Y origin field. **LOZORIG** (N) Implicit local Z origin
  field. **ADJUST** (N) Numeric field to identify final or temporary coordinates.
  Required=Yes

### Fields:

### Parameters:

* **calctype**:
  Numeric flag to identify the method of coordinate computation to be used: 0 =
  Automatic. 1 = Bearing and distance method. 2 = Angle and distance method. 3 = Resection
  by angles. 4 = Distance intersection. 5 = Bearing intersection. 6 = Angle intersection.
  7 = Resection by distances. The default is the automatic method (0).
  Range=0,7
  Values=0,1,2,3,4,5,6,7
  Default=0
  Required=Yes

* **angle**:
  Units of angle measurements : 1 = Degrees, minutes and seconds. [0-360] in the form
  DDD.MMSS 2 = Gradians. [0-400] The default angle unit is degrees, minutes and seconds
  (1).
  Range=1,2
  Values=1,2
  Default=1
  Required=No

* **recalc**:
  Optional numeric flag to allow recalculation of stations established from a resurveyed
  base line. The default is not to have recalculation (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **weight**:
  Optional flag numeric to allow assignment of weights proportional to the error values
  in the input file. The default is not to have weights assignment (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **adjust**:
  Optional numeric flag to allow coordinate adjustment where redundant measurements
  occur. The adjustment is done by the least square error method. The default is to have
  coordinates computed from a single set of measurements (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **ndec**:
  Number of decimals in the output results summary. Only used if parameter PRINT is set
  to one (2).
  Range=Undefined
  Values=Undefined
  Default=2
  Required=No

* **print**:
  Set to one to display a summary of results to the screen and print file. The default is
  not to print summary results (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('surcal')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute surcal
print("Running surcal...")
dm_cmd.surcal(
    out_o='t_surcal_out',  # required
    calctype_p='required_val',  # required
    # in_i='t_assays',  # optional
    # angle_p=1,  # optional
    # recalc_p=0,  # optional
    # weight_p=0,  # optional
    # adjust_p=0,  # optional
    # ndec_p=2,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("surcal execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_surcal_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")